In [4]:
pip install ultralytics opencv-python-headless pywavelets matplotlib pandas torch torchvision open_clip_torch


   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   - -------------------------------------- 1.0/38.9 MB 5.6 MB/s eta 0:00:07
   - -------------------------------------- 1.6/38.9 MB 3.6 MB/s eta 0:00:11
   - -------------------------------------- 1.8/38.9 MB 2.7 MB/s eta 0:00:14
   -- ------------------------------------- 2.1/38.9 MB 2.4 MB/s eta 0:00:16
   -- ------------------------------------- 2.4/38.9 MB 2.3 MB/s eta 0:00:17
   -- ------------------------------------- 2.4/38.9 MB 2.3 MB/s eta 0:00:17
   -- ------------------------------------- 2.4/38.9 MB 2.3 MB/s eta 0:00:17
   -- ------------------------------------- 2.6/38.9 MB 1.5 MB/s eta 0:00:25
   -- ------------------------------------- 2.9/38.9 MB 1.4 MB/s eta 0:00:26
   --- ------------------------------------ 3.1/38.9 MB 1.4 MB/s eta 0:00:25
   --- ------------------------------------ 3.7/38.9 MB 1.4 MB/s eta 0:00:25
   ---- ----------------------------------- 4.2/38.9 MB 1.5 MB/s eta 0:00:24
   ---

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\paruv\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [6]:
pip install open_clip_torch


  Using cached open_clip_torch-3.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached timm-1.0.20-py3-none-any.whl.metadata (61 kB)
Using cached open_clip_torch-3.2.0-py3-none-any.whl (1.5 MB)
Using cached timm-1.0.20-py3-none-any.whl (2.5 MB)
Using cached ftfy-6.3.1-py3-none-any.whl (44 kB)

   ---------------------------------------- 0/3 [ftfy]
   ---------------------------------------- 0/3 [ftfy]
   ---------------------------------------- 0/3 [ftfy]
   ---------------------------------------- 0/3 [ftfy]
   ---------------------------------------- 0/3 [ftfy]
   ---------------------------------------- 0/3 [ftfy]
   ------------- -------------------------- 1/3 [timm]
   ------------- -------------------------- 1/3 [timm]
   ------------- -------------------------- 1/3 [timm]
   ------------- -------------------------- 1/3 [timm]
   ------------- -------------------------- 1/3 [timm]
   ------------- -----------------------

In [7]:
import open_clip
print(open_clip.__version__)


c:\Users\paruv\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3.2.0


In [11]:
# ===========================================================
# CCTV Household Object Detection (No-Person Dataset)
# YOLOv8 + CLIP Hybrid Model (Custom + General Detection)
# ===========================================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pywt
import pandas as pd
import torch
import open_clip
from ultralytics import YOLO
from PIL import Image

# ===========================================================
# Step 1: Setup Paths and Models
# ===========================================================

dataset_path = "dataset/"
output_path = "output_full/"
report_path = os.path.join(output_path, "classification_report.csv")
os.makedirs(output_path, exist_ok=True)

image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
if len(image_files) == 0:
    raise FileNotFoundError("❌ No images found in 'dataset/' folder.")
print(f"=== Found {len(image_files)} images ===")

# Load YOLOv8x model
print("=== Loading YOLOv8x Model ===")
yolo_model = YOLO("yolov8x.pt")

# Load CLIP model for zero-shot classification
print("=== Loading CLIP Model ===")
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(device)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

# ===========================================================
# Step 2: Custom Object Labels
# ===========================================================

custom_labels = [
    "cloth", "mirror", "air conditioner", "window", "towel", "shower", "cupboard", "bedsheet",
    "air cooler", "pillow", "table", "dustbin", "glass", "table fan", "switch", "curtain",
    "building", "tree", "dream catcher", "cap", "doll", "spoon", "cover", "charger",
    "spectacle", "box", "chocolates", "AC ventilator", "steel item", "balcony", "mountain",
    "door", "mop", "washing machine", "wall", "soap", "photo frame", "lamp", "slipper",
    "shoe", "boots", "weight machine", "drawer", "statue", "staircase", "light", "basketball court",
    "swimming pool", "telephone","Extension box","table tennis table", "badminton court", "stadium"
]
text_inputs = tokenizer(custom_labels).to(device)
with torch.no_grad():
    text_features = clip_model.encode_text(text_inputs)
    text_features /= text_features.norm(dim=-1, keepdim=True)

# ===========================================================
# Step 3: Image Enhancement
# ===========================================================

def enhance_image_color(img):
    """Low-light enhancement optimized for CCTV images."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    enhanced_lab = cv2.merge((cl, a, b))
    clahe_img = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

    hsv = cv2.cvtColor(clahe_img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    v = np.clip(v * 1.3, 0, 255).astype(np.uint8)
    bright_img = cv2.cvtColor(cv2.merge([h, s, v]), cv2.COLOR_HSV2BGR)

    gray = cv2.cvtColor(bright_img, cv2.COLOR_BGR2GRAY)
    coeffs = pywt.wavedec2(gray, 'db1', level=2)
    coeffs_H = list(coeffs)
    coeffs_H[0] *= 1.2
    wavelet_boost = pywt.waverec2(coeffs_H, 'db1')
    wavelet_boost = np.clip(wavelet_boost, 0, 255).astype(np.uint8)
    wavelet_boost = cv2.resize(wavelet_boost, (bright_img.shape[1], bright_img.shape[0]))

    enhanced_final = cv2.addWeighted(
        bright_img, 0.85, cv2.cvtColor(wavelet_boost, cv2.COLOR_GRAY2BGR), 0.15, 0
    )
    return enhanced_final

# ===========================================================
# Step 4: YOLO + CLIP Detection
# ===========================================================

def detect_objects_hybrid(enhanced_img):
    """Combine YOLO object detection + CLIP zero-shot custom classification."""
    output_img = enhanced_img.copy()
    all_detections = []

    # Run YOLO detection
    results = yolo_model.predict(source=enhanced_img, conf=0.25, iou=0.45, verbose=False)
    boxes = results[0].boxes
    names = yolo_model.names

    for box in boxes:
        cls_id = int(box.cls[0])
        label_yolo = names[cls_id]
        conf_yolo = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].int().tolist()

        if label_yolo.lower() == "person":
            continue  # skip person class

        crop = enhanced_img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        # --- CLIP classification on cropped region ---
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        crop_pil = Image.fromarray(crop_rgb)
        image_tensor = preprocess(crop_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = clip_model.encode_image(image_tensor)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            similarity = (100.0 * image_features @ text_features.T)
            probs = similarity.softmax(dim=-1).cpu().numpy()[0]

        top_idx = np.argmax(probs)
        label_clip = custom_labels[top_idx]
        conf_clip = float(probs[top_idx] * 100)

        # Combine labels
        final_label = label_clip if conf_clip > 35 else label_yolo
        final_conf = max(conf_yolo * 100, conf_clip)

        # Draw box and text
        cv2.rectangle(output_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output_img, f"{final_label} ({final_conf:.1f}%)",
                    (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0, 255, 0), 2)

        all_detections.append({
            "YOLO_Label": label_yolo,
            "CLIP_Label": label_clip,
            "Final_Label": final_label,
            "Confidence(%)": round(final_conf, 2)
        })

    return output_img, all_detections

# ===========================================================
# Step 5: Process Dataset
# ===========================================================

report_data = []
for idx, image_name in enumerate(image_files, 1):
    print(f"\n=== [{idx}/{len(image_files)}] Processing: {image_name} ===")
    image_path = os.path.join(dataset_path, image_name)
    img = cv2.imread(image_path)
    if img is None:
        print(f"⚠️ Skipping {image_name}")
        continue

    enhanced_img = enhance_image_color(img)
    detected_img, detected_objects = detect_objects_hybrid(enhanced_img)

    name_no_ext = os.path.splitext(image_name)[0]
    cv2.imwrite(os.path.join(output_path, f"{name_no_ext}_enhanced.png"), enhanced_img)
    cv2.imwrite(os.path.join(output_path, f"{name_no_ext}_detected.png"), detected_img)

    if detected_objects:
        for obj in detected_objects:
            report_data.append({
                "Image": image_name,
                **obj,
                "Total_Detections": len(detected_objects)
            })
        print(f"✅ Detected: {', '.join([d['Final_Label'] for d in detected_objects])}")
    else:
        report_data.append({
            "Image": image_name,
            "YOLO_Label": "None",
            "CLIP_Label": "None",
            "Final_Label": "None",
            "Confidence(%)": 0,
            "Total_Detections": 0
        })
        print("⚠️ No objects detected.")

# ===========================================================
# Step 6: Save Report
# ===========================================================

df = pd.DataFrame(report_data)
df.to_csv(report_path, index=False)
print(f"\n📊 Classification report saved at: {report_path}")

# ===========================================================
# Step 7: Visualization
# ===========================================================

sample_images = np.random.choice(image_files, min(3, len(image_files)), replace=False)
fig, axes = plt.subplots(len(sample_images), 3, figsize=(12, 4 * len(sample_images)))
fig.suptitle("YOLOv8 + CLIP Hybrid Detection (Custom + COCO)", fontsize=16)

for idx, image_name in enumerate(sample_images):
    orig = cv2.imread(os.path.join(dataset_path, image_name))
    enh = cv2.imread(os.path.join(output_path, f"{os.path.splitext(image_name)[0]}_enhanced.png"))
    det = cv2.imread(os.path.join(output_path, f"{os.path.splitext(image_name)[0]}_detected.png"))

    axes[idx, 0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
    axes[idx, 0].set_title("Original")
    axes[idx, 1].imshow(cv2.cvtColor(enh, cv2.COLOR_BGR2RGB))
    axes[idx, 1].set_title("Enhanced")
    axes[idx, 2].imshow(cv2.cvtColor(det, cv2.COLOR_BGR2RGB))
    axes[idx, 2].set_title("Hybrid Detected")

    for ax in axes[idx]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("\n🎯 Completed! Results saved in 'output_full/' folder.")


=== Found 485 images ===
=== Loading YOLOv8x Model ===
=== Loading CLIP Model ===

=== [1/485] Processing: 10.png ===
✅ Detected: doll, teddy bear, bed, teddy bear, pillow, teddy bear

=== [2/485] Processing: 100.png ===
✅ Detected: bowl, bowl, bowl, table fan

=== [3/485] Processing: 101.png ===
✅ Detected: bottle, chair, bowl, chair, chair, sink

=== [4/485] Processing: 102.png ===
✅ Detected: bottle, bowl, chair, chair, chair, sink

=== [5/485] Processing: 103.png ===
✅ Detected: bottle, chair, sink, chair, chair

=== [6/485] Processing: 104.png ===
✅ Detected: bottle, sink

=== [7/485] Processing: 105.png ===
✅ Detected: bowl, bottle, bowl, bottle, bowl, bottle, bottle

=== [8/485] Processing: 106.png ===
✅ Detected: bowl, bottle, bottle, bowl, bowl, bottle, bottle

=== [9/485] Processing: 107.png ===
✅ Detected: bowl, bowl, bowl, bottle, cup

=== [10/485] Processing: 109.png ===
⚠️ No objects detected.

=== [11/485] Processing: 110.png ===
⚠️ No objects detected.

=== [12/485] Pro

KeyboardInterrupt: 

In [12]:
# ===========================================================
# Step 8: Detection Accuracy Metrics (Confidence-Based)
# ===========================================================

import numpy as np
import pandas as pd
import os
import cv2

print("\n=== 🎯 Calculating Detection Accuracy Metrics ===")

total_images = len(image_files)
images_with_detections = 0
all_confidences = []

for idx, image_name in enumerate(image_files, 1):
    image_path = os.path.join(dataset_path, image_name)
    img = cv2.imread(image_path)
    if img is None:
        print(f"⚠️ Skipping {image_name} (unreadable)")
        continue

    # Step 1: Enhance + Detect
    enhanced_img = enhance_image_color(img)
    _, detected_objects = detect_objects_hybrid(enhanced_img)

    # Step 2: Count valid detections
    if len(detected_objects) > 0:
        images_with_detections += 1
        # use correct key name for confidence
        all_confidences.extend([obj["Confidence(%)"] for obj in detected_objects])

    print(f"[{idx}/{total_images}] Processed: {image_name} | "
          f"Detected: {len(detected_objects)} objects")

# Step 3: Metrics calculation
detection_rate = (images_with_detections / total_images) * 100 if total_images else 0
avg_conf = np.mean(all_confidences) if len(all_confidences) > 0 else 0
relative_accuracy = (detection_rate * (avg_conf / 100))  # Combined overall score

# Step 4: Display metrics
print("\n=== 📊 Detection Accuracy Metrics ===")
print(f"Total Images Tested        : {total_images}")
print(f"Images with Detections     : {images_with_detections}")
print(f"Detection Rate (%)         : {detection_rate:.2f}")
print(f"Average Confidence (%)     : {avg_conf:.2f}")
print(f"Overall Relative Accuracy  : {relative_accuracy:.2f}")

# Step 5: Save metrics to CSV
acc_data = {
    "Total_Images": [total_images],
    "Images_with_Detections": [images_with_detections],
    "Detection_Rate(%)": [round(detection_rate, 2)],
    "Avg_Confidence(%)": [round(avg_conf, 2)],
    "Relative_Accuracy(%)": [round(relative_accuracy, 2)]
}

os.makedirs("output_full", exist_ok=True)
df_acc = pd.DataFrame(acc_data)
acc_path = os.path.join("output_full", "detection_accuracy_metrics.csv")
df_acc.to_csv(acc_path, index=False)

print(f"\n✅ Accuracy metrics saved to '{acc_path}'")



=== 🎯 Calculating Detection Accuracy Metrics ===
[1/485] Processed: 10.png | Detected: 6 objects
[2/485] Processed: 100.png | Detected: 4 objects
[3/485] Processed: 101.png | Detected: 6 objects
[4/485] Processed: 102.png | Detected: 6 objects
[5/485] Processed: 103.png | Detected: 5 objects
[6/485] Processed: 104.png | Detected: 2 objects
[7/485] Processed: 105.png | Detected: 7 objects
[8/485] Processed: 106.png | Detected: 7 objects
[9/485] Processed: 107.png | Detected: 5 objects
[10/485] Processed: 109.png | Detected: 0 objects
[11/485] Processed: 110.png | Detected: 0 objects
[12/485] Processed: 112.png | Detected: 0 objects
[13/485] Processed: 113.png | Detected: 0 objects
[14/485] Processed: 114.png | Detected: 0 objects
[15/485] Processed: 115.png | Detected: 2 objects
[16/485] Processed: 116.png | Detected: 3 objects
[17/485] Processed: 117.png | Detected: 5 objects
[18/485] Processed: 118.png | Detected: 5 objects
[19/485] Processed: 119.png | Detected: 7 objects
[20/485] P

In [4]:
import zipfile, os, cv2, pandas as pd
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr

dataset_zip = "dataset.zip"
output_zip = "output_full.zip"
dataset_path, output_path = "dataset", "output_full"

# Extract
for z, p in [(dataset_zip, dataset_path), (output_zip, output_path)]:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall(p)

def calc_iou(a, b):
    inter = cv2.bitwise_and(a, b)
    union = cv2.bitwise_or(a, b)
    u = cv2.countNonZero(union)
    return cv2.countNonZero(inter) / u if u else 0

results = []
for name in os.listdir(dataset_path):
    if not name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
    b = os.path.splitext(name)[0]
    epath = os.path.join(output_path, f"{b}_enhanced.png")
    dpath = os.path.join(output_path, f"{b}_detected.png")
    if not (os.path.exists(epath) and os.path.exists(dpath)): continue
    e, d = cv2.imread(epath), cv2.imread(dpath)
    if e is None or d is None: continue

    ge, gd = cv2.cvtColor(e, cv2.COLOR_BGR2GRAY), cv2.cvtColor(d, cv2.COLOR_BGR2GRAY)
    if ge.shape != gd.shape:
        gd = cv2.resize(gd, (ge.shape[1], ge.shape[0]))

    results.append({
        "Image": name,
        "IoU": round(calc_iou(ge, gd), 3),
        "PSNR": round(psnr(ge, gd, data_range=255), 2),
        "SSIM": round(ssim(ge, gd, data_range=255), 3)
    })

df = pd.DataFrame(results)
df.to_csv("image_quality_metrics.csv", index=False)
print(df)


c:\Users\paruv\AppData\Local\Programs\Python\Python312\Lib\site-packages\skimage\metrics\simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


       Image    IoU   PSNR   SSIM
0    100.png  0.991  19.99  0.904
1    102.png  0.995  21.12  0.903
2    103.png  0.984  19.41  0.879
3    104.png  0.999  26.66  0.968
4    105.png  0.994  19.02  0.834
..       ...    ...    ...    ...
154   47.png  0.996  23.68  0.953
155  470.png  0.991  18.09  0.815
156  471.png  0.999  25.84  0.967
157  472.png  0.998  20.99  0.912
158  473.png  0.998  20.68  0.900

[159 rows x 4 columns]
